In [ ]:
!pip install langchain sentence-transformers pypdf langchain-text-splitters langchain-community langchain-huggingface

In [ ]:
!pip install faiss-cpu

In [ ]:
# @title Install components
!sudo apt-get install zstd

!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama -q

import os
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'

# Start Ollama server in background
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print("Done ✓")

In [ ]:
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# ── 3. Start Ollama server in background ─────────────────────────────────────
import subprocess, time, requests

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env=os.environ.copy()
)

# Wait until server is ready
print("Starting Ollama server", end="")
for _ in range(20):
    try:
        r = requests.get("http://localhost:11434")
        if r.status_code == 200:
            print(" ✓ Ready!")
            break
    except:
        print(".", end="", flush=True)
        time.sleep(1.5)

# ── 4. Pull a model ───────────────────────────────────────────────────────────
# Good T4-friendly models (4GB VRAM safe):
#   llama3.2        ~2GB
#   mistral         ~4GB
#   gemma2:2b       ~1.5GB
#   phi3:mini       ~2.3GB

MODEL = "llama3.2"  # change this to your preferred model
print(f"\nPulling {MODEL}...")
!ollama pull {MODEL}

# ── 5. Verify GPU is being used ───────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.used,memory.free --format=csv

In [ ]:
!ollama pull {"gemma2:2b"}
!ollama pull {"mistral"}

# RAG

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("IPO_SGTM.pdf")
documents = loader.load()

print(len(documents))  # number of pages

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)
print(len(docs))

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
import ollama

# -----------------------------
# 1. Define models (agents)
# -----------------------------
MODELS = {
    "Financial Analyst": "gemma2:2b",
    "Risk Analyst": "llama3.2",
    "Auditor": "mistral"
}

# -----------------------------
# 2. Ask model
# -----------------------------
def ask_model(model_name, prompt):
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    content = response["message"]["content"]
    print(f"\n🧠 {model_name} response:\n{content}\n")
    return content


# -----------------------------
# 3. Personas (STRONG prompts)
# -----------------------------
PERSONAS = {
    "Financial Analyst": """You are a professional Financial Analyst.

Instructions:
- Analyze the question using the provided context
""",

    "Risk Analyst": """You are a Risk Analyst.

Instructions:
- Critically evaluate the financial situation
""",

    "Auditor": """You are a Financial Auditor.

Instructions:
- Verify the answer strictly using the context
"""
}


# -----------------------------
# 4. Initial answers
# -----------------------------
def generate_initial_answers(question, context):
    answers = {}

    for persona_name, model in MODELS.items():
        print(f"🔹 Asking {persona_name}...")

        prompt = f"""
{PERSONAS[persona_name]}

Context:
{context}

Question:
{question}
"""

        answers[persona_name] = ask_model(model, prompt)

    return answers


# -----------------------------
# 5. Debate step
# -----------------------------
def debate_step(question, context, answers):
    improved = {}

    for persona_name, model in MODELS.items():
        other_answers = "\n\n".join(
            [f"{k}: {v}" for k, v in answers.items() if k != persona_name]
        )

        prompt = f"""
You are {persona_name}.

Context:
{context}

Question:
{question}

Other answers:
{other_answers}

Your task:
1. Critique other answers
2. Identify mistakes
3. Improve your answer

Return ONLY your improved final answer.
"""

        print(f"🧠 {persona_name} debating...")
        improved[persona_name] = ask_model(model, prompt)

    return improved


# -----------------------------
# 6. Judge step (FIXED 🔥)
# -----------------------------
def judge_answers(question, context, answers):
    combined = "\n\n".join([f"{k}: {v}" for k, v in answers.items()])

    judge_prompt = f"""
You are a financial judge.

Context:
{context}

Question:
{question}

Answers:
{combined}

Evaluate based on:
- correctness (based on context)
- reasoning quality
- consistency with documents

Return:
1. Best answer
2. Which persona is correct
3. Why
"""

    return ask_model("mistral", judge_prompt)


# -----------------------------
# 7. Full pipeline
# -----------------------------
def multi_llm_debate(question, context):
    print("\n=== Initial Answers ===")
    answers = generate_initial_answers(question, context)

    print("\n=== Debate Round ===")
    improved_answers = debate_step(question, context, answers)

    print("\n=== Judge Decision ===")
    final = judge_answers(question, context, improved_answers)

    return final


# -----------------------------
# Run example
# -----------------------------
if __name__ == "__main__":
    question = "Do new investors receive dividends for 2024?"

    docs = vectorstore.similarity_search(question, k=3)
    context = "\n".join([doc.page_content for doc in docs])

    result = multi_llm_debate(question, context)

    print("\n🏆 FINAL RESULT:\n", result)